In [1]:
from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nFiles in this directory:")
for file_path in Path.cwd().iterdir():
    print("-", file_path.name)

Current directory:
/home/xilinx/jupyter_notebooks/RV_PYNQ

Files in this directory:
- test_sort.hex
- design.bit
- design.hwh
- verify.ipynb
- .ipynb_checkpoints


In [2]:
from pathlib import Path

required_files = [
    "design.bit",
    "design.hwh",
    "test_sort.hex",
]

missing_files = [
    filename
    for filename in required_files
    if not Path(filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files: "
        + ", ".join(missing_files)
    )

print("File structure check passed.")
print("All required project files are present.")

File structure check passed.
All required project files are present.


In [3]:
from pynq import Overlay

BITSTREAM_PATH = str(Path.cwd() / "design.bit")

print("Loading:", BITSTREAM_PATH)

overlay = Overlay(BITSTREAM_PATH)

print("Overlay loaded successfully.")

Loading: /home/xilinx/jupyter_notebooks/RV_PYNQ/design.bit


Overlay loaded successfully.


In [4]:
print("IP blocks found in design.hwh:\n")

for ip_name, information in overlay.ip_dict.items():
    print(ip_name)

    physical_address = information.get("phys_addr")
    address_range = information.get("addr_range")

    if physical_address is not None:
        print(f"  Base address: 0x{physical_address:08X}")

    if address_range is not None:
        print(f"  Address range: 0x{address_range:X}")

IP blocks found in design.hwh:

axi_gpio_0
  Base address: 0x41200000
  Address range: 0x20000
processing_system7_0


In [5]:
from pathlib import Path

ROOT = Path.cwd()

BITSTREAM_PATH = ROOT / "design.bit"
HWH_PATH = ROOT / "design.hwh"
HEX_PATH = ROOT / "test_sort.hex"

In [2]:
from pathlib import Path
import time
import random
import re

from pynq import Overlay, MMIO

print("Python modules imported successfully.")

Python modules imported successfully.


In [3]:
ROOT = Path.cwd()

BITSTREAM_PATH = ROOT / "design.bit"
HWH_PATH = ROOT / "design.hwh"
HEX_PATH = ROOT / "test_sort.hex"

print("Current directory:")
print(ROOT)

print("\nProject files:")

for file_path in [
    BITSTREAM_PATH,
    HWH_PATH,
    HEX_PATH,
]:
    status = "FOUND" if file_path.exists() else "MISSING"
    print(f"{file_path.name:20s} {status}")

Current directory:
/home/xilinx/jupyter_notebooks/RV_PYNQ

Project files:
design.bit           FOUND
design.hwh           FOUND
test_sort.hex        FOUND


In [4]:
required_files = [
    BITSTREAM_PATH,
    HWH_PATH,
    HEX_PATH,
]

missing_files = [
    file_path.name
    for file_path in required_files
    if not file_path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files: "
        + ", ".join(missing_files)
    )

print("File check passed.")

File check passed.


In [5]:
print("Loading FPGA bitstream...")

overlay = Overlay(str(BITSTREAM_PATH))

print("Overlay object created.")

if overlay.is_loaded():
    print("Bitstream loaded successfully.")
else:
    raise RuntimeError("Bitstream was not loaded.")

Loading FPGA bitstream...


Overlay object created.
Bitstream loaded successfully.


In [6]:
print("Addressable IP blocks:\n")

for ip_name, information in overlay.ip_dict.items():
    physical_address = information.get("phys_addr")
    address_range = information.get("addr_range")
    ip_type = information.get("type", "Unknown")

    print(f"IP name: {ip_name}")
    print(f"  Type: {ip_type}")

    if physical_address is not None:
        print(f"  Base address: 0x{physical_address:08X}")

    if address_range is not None:
        print(f"  Address range: 0x{address_range:X}")

    print()

Addressable IP blocks:

IP name: axi_gpio_0
  Type: xilinx.com:ip:axi_gpio:2.0
  Base address: 0x41200000
  Address range: 0x20000

IP name: processing_system7_0
  Type: xilinx.com:ip:processing_system7:5.5



In [7]:
EXPECTED_ADDRESSES = {
    0x40000000: "Instruction BRAM",
    0x41200000: "Reset GPIO",
    0x42000000: "Data BRAM",
}

detected_addresses = {}

for ip_name, information in overlay.ip_dict.items():
    address = information.get("phys_addr")

    if address is not None:
        detected_addresses[address] = ip_name


print("Address verification:\n")

all_addresses_found = True

for expected_address, purpose in EXPECTED_ADDRESSES.items():
    if expected_address in detected_addresses:
        print(
            f"PASS: {purpose:20s} "
            f"0x{expected_address:08X} "
            f"({detected_addresses[expected_address]})"
        )
    else:
        print(
            f"FAIL: {purpose:20s} "
            f"0x{expected_address:08X} not found"
        )

        all_addresses_found = False


Address verification:

FAIL: Instruction BRAM     0x40000000 not found
PASS: Reset GPIO           0x41200000 (axi_gpio_0)
FAIL: Data BRAM            0x42000000 not found


In [8]:
from pprint import pprint

print("=" * 70)
print("overlay.ip_dict")
print("=" * 70)
pprint(overlay.ip_dict)

print("\n" + "=" * 70)
print("overlay.mem_dict")
print("=" * 70)

memory_dictionary = getattr(overlay, "mem_dict", {})

if memory_dictionary:
    pprint(memory_dictionary)
else:
    print("mem_dict is empty or unavailable.")

overlay.ip_dict
{'axi_gpio_0': {'addr_range': 131072,
                'bdtype': None,
                'device': <pynq.pl_server.embedded_device.EmbeddedDevice object at 0xb38c18f8>,
                'driver': <class 'pynq.lib.axigpio.AxiGPIO'>,
                'fullpath': 'axi_gpio_0',
                'gpio': {},
                'interrupts': {},
                'mem_id': 'S_AXI',
                'memtype': 'REGISTER',
                'parameters': {'ADDR_WIDTH': '9',
                               'ARUSER_WIDTH': '0',
                               'AWUSER_WIDTH': '0',
                               'BUSER_WIDTH': '0',
                               'CLK_DOMAIN': 'axi_bram_bd_processing_system7_0_0_FCLK_CLK0',
                               'C_ALL_INPUTS': '0',
                               'C_ALL_INPUTS_2': '0',
                               'C_ALL_OUTPUTS': '1',
                               'C_ALL_OUTPUTS_2': '0',
                               'C_BASEADDR': '0x41200000',
       

In [9]:
def get_base_address(info):
    """
    Try common field names used by different PYNQ versions.
    """
    for key in (
        "phys_addr",
        "base_address",
        "base_addr",
        "address",
    ):
        value = info.get(key)

        if isinstance(value, int):
            return value

    return None


def get_address_range(info):
    """
    Try common address-range field names.
    """
    for key in (
        "addr_range",
        "address_range",
        "range",
        "size",
    ):
        value = info.get(key)

        if isinstance(value, int):
            return value

    return None


address_regions = []

metadata_sources = {
    "ip_dict": overlay.ip_dict,
    "mem_dict": getattr(overlay, "mem_dict", {}),
}

for source_name, dictionary in metadata_sources.items():
    for item_name, information in dictionary.items():
        if not isinstance(information, dict):
            continue

        base_address = get_base_address(information)
        address_range = get_address_range(information)

        if base_address is not None:
            address_regions.append({
                "source": source_name,
                "name": item_name,
                "base": base_address,
                "range": address_range,
            })


print("Detected address regions:\n")

for region in sorted(
    address_regions,
    key=lambda item: item["base"]
):
    range_text = (
        f"0x{region['range']:X}"
        if region["range"] is not None
        else "unknown"
    )

    print(
        f"{region['source']:10s} "
        f"{region['name']:35s} "
        f"base=0x{region['base']:08X} "
        f"range={range_text}"
    )

Detected address regions:

mem_dict   PSDDR                               base=0x00000000 range=0x10000000
mem_dict   axi_bram_ctrl_0                     base=0x40000000 range=0x2000
ip_dict    axi_gpio_0                          base=0x41200000 range=0x20000
mem_dict   axi_bram_ctrl_1                     base=0x42000000 range=0x2000


In [10]:
EXPECTED_ADDRESSES = {
    0x40000000: "Instruction BRAM",
    0x41200000: "Reset GPIO",
    0x42000000: "Data BRAM",
}

detected_by_address = {
    region["base"]: region
    for region in address_regions
}

print("Address metadata verification:\n")

for expected_address, purpose in EXPECTED_ADDRESSES.items():
    region = detected_by_address.get(expected_address)

    if region is not None:
        print(
            f"PASS: {purpose:20s} "
            f"0x{expected_address:08X} "
            f"({region['source']} → {region['name']})"
        )
    else:
        print(
            f"NOT LISTED: {purpose:20s} "
            f"0x{expected_address:08X}"
        )

Address metadata verification:

PASS: Instruction BRAM     0x40000000 (mem_dict → axi_bram_ctrl_0)
PASS: Reset GPIO           0x41200000 (ip_dict → axi_gpio_0)
PASS: Data BRAM            0x42000000 (mem_dict → axi_bram_ctrl_1)


In [11]:
from pynq import MMIO
import time

INSTR_BASE = 0x40000000
RESET_BASE = 0x41200000
DATA_BASE  = 0x42000000

INSTR_RANGE = 0x1000
RESET_RANGE = 0x1000

# Must include data offset 0x0000 and DONE offset 0x1000
DATA_RANGE = 0x2000

instr_mmio = MMIO(INSTR_BASE, INSTR_RANGE)
reset_mmio = MMIO(RESET_BASE, RESET_RANGE)
data_mmio  = MMIO(DATA_BASE, DATA_RANGE)

print("All MMIO objects were created.")

All MMIO objects were created.


In [12]:
CPU_STOP = 0x00000000

reset_mmio.write(0x00, CPU_STOP)
time.sleep(0.01)

print("CPU is held in reset.")

CPU is held in reset.


In [13]:
TEST_PATTERN = 0xA5A55A5A
TEST_OFFSET = 0x0000

original_data_word = int(
    data_mmio.read(TEST_OFFSET)
) & 0xFFFFFFFF

print(f"Original Data BRAM word: 0x{original_data_word:08X}")

try:
    data_mmio.write(TEST_OFFSET, TEST_PATTERN)

    data_readback = int(
        data_mmio.read(TEST_OFFSET)
    ) & 0xFFFFFFFF

    print(f"Written pattern:         0x{TEST_PATTERN:08X}")
    print(f"Read-back value:         0x{data_readback:08X}")

    if data_readback != TEST_PATTERN:
        raise RuntimeError(
            "Data BRAM MMIO test failed."
        )

    print("PASS: Data BRAM is accessible at 0x42000000.")

finally:
    data_mmio.write(TEST_OFFSET, original_data_word)

    restored_value = int(
        data_mmio.read(TEST_OFFSET)
    ) & 0xFFFFFFFF

    print(f"Restored original word:  0x{restored_value:08X}")

Original Data BRAM word: 0x00000000
Written pattern:         0xA5A55A5A
Read-back value:         0xA5A55A5A
PASS: Data BRAM is accessible at 0x42000000.
Restored original word:  0x00000000


In [14]:
TEST_PATTERN = 0x12345678
TEST_OFFSET = 0x0000

original_instruction = int(
    instr_mmio.read(TEST_OFFSET)
) & 0xFFFFFFFF

print(
    f"Original Instruction BRAM word: "
    f"0x{original_instruction:08X}"
)

try:
    instr_mmio.write(TEST_OFFSET, TEST_PATTERN)

    instruction_readback = int(
        instr_mmio.read(TEST_OFFSET)
    ) & 0xFFFFFFFF

    print(f"Written pattern:                0x{TEST_PATTERN:08X}")
    print(f"Read-back value:                0x{instruction_readback:08X}")

    if instruction_readback != TEST_PATTERN:
        raise RuntimeError(
            "Instruction BRAM MMIO test failed."
        )

    print(
        "PASS: Instruction BRAM is accessible "
        "at 0x40000000."
    )

finally:
    instr_mmio.write(
        TEST_OFFSET,
        original_instruction
    )

    restored_instruction = int(
        instr_mmio.read(TEST_OFFSET)
    ) & 0xFFFFFFFF

    print(
        f"Restored original instruction:  "
        f"0x{restored_instruction:08X}"
    )

Original Instruction BRAM word: 0x00000000
Written pattern:                0x12345678
Read-back value:                0x12345678
PASS: Instruction BRAM is accessible at 0x40000000.
Restored original instruction:  0x00000000


In [15]:
print("HEX file:", HEX_PATH)
print()

with open(HEX_PATH, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file):
        print(f"{line_number + 1:3d}: {line.rstrip()}")

        if line_number >= 9:
            break

HEX file: /home/xilinx/jupyter_notebooks/RV_PYNQ/test_sort.hex

  1: 000012B7
  2: 00000313
  3: 01F00393
  4: 02735E63
  5: 00000E13
  6: 40638EB3
  7: 00028F33
  8: 03DE5263
  9: 000F2F83
 10: 004F2503


In [16]:
import re


def load_hex_words(hex_path):
    """
    Read 32-bit machine-code words from a HEX text file.
    按照低地址到高地址的顺序读取机器码。
    """
    words = []

    with open(hex_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            # Remove comments
            line = line.split("//", 1)[0]
            line = line.split("#", 1)[0]
            line = line.strip()

            if not line:
                continue

            # Ignore Verilog-style address markers
            if line.startswith("@"):
                continue

            line = line.replace("_", "")
            line = line.removeprefix("0x")
            line = line.removeprefix("0X")

            if not re.fullmatch(r"[0-9A-Fa-f]{8}", line):
                raise ValueError(
                    f"Invalid HEX word at line {line_number}: {line!r}"
                )

            words.append(int(line, 16))

    if not words:
        raise ValueError("No machine-code words were found.")

    return words

In [17]:
PROGRAM_WORDS = load_hex_words(HEX_PATH)

print(f"Loaded {len(PROGRAM_WORDS)} instructions.")

for index, word in enumerate(PROGRAM_WORDS):
    print(f"PC=0x{index * 4:04X}: 0x{word:08X}")

Loaded 23 instructions.
PC=0x0000: 0x000012B7
PC=0x0004: 0x00000313
PC=0x0008: 0x01F00393
PC=0x000C: 0x02735E63
PC=0x0010: 0x00000E13
PC=0x0014: 0x40638EB3
PC=0x0018: 0x00028F33
PC=0x001C: 0x03DE5263
PC=0x0020: 0x000F2F83
PC=0x0024: 0x004F2503
PC=0x0028: 0x01F55663
PC=0x002C: 0x00AF2023
PC=0x0030: 0x01FF2223
PC=0x0034: 0x004F0F13
PC=0x0038: 0x001E0E13
PC=0x003C: 0xFE1FF06F
PC=0x0040: 0x00130313
PC=0x0044: 0xFC9FF06F
PC=0x0048: 0xCAFEC5B7
PC=0x004C: 0xABE58593
PC=0x0050: 0x00002637
PC=0x0054: 0x00B62023
PC=0x0058: 0x0000006F


In [19]:
def write_words(mmio, start_offset, words):
    for index, word in enumerate(words):
        offset = start_offset + index * 4
        mmio.write(offset, int(word) & 0xFFFFFFFF)


def read_words(mmio, start_offset, count):
    result = []

    for index in range(count):
        offset = start_offset + index * 4
        word = int(mmio.read(offset)) & 0xFFFFFFFF
        result.append(word)

    return result

In [20]:
# Keep CPU stopped while changing instruction memory
reset_mmio.write(0x00, CPU_STOP)
time.sleep(0.01)

write_words(
    instr_mmio,
    start_offset=0x0000,
    words=PROGRAM_WORDS
)

print(
    f"Wrote {len(PROGRAM_WORDS)} instructions "
    "to Instruction BRAM."
)

Wrote 23 instructions to Instruction BRAM.


In [21]:
program_readback = read_words(
    instr_mmio,
    start_offset=0x0000,
    count=len(PROGRAM_WORDS)
)

if program_readback == PROGRAM_WORDS:
    print("PASS: Program write/readback verification succeeded.")

else:
    print("FAIL: Program readback mismatch.")

    for index, (expected, actual) in enumerate(
        zip(PROGRAM_WORDS, program_readback)
    ):
        if expected != actual:
            print(f"First mismatch at instruction {index}")
            print(f"Address: 0x{INSTR_BASE + index * 4:08X}")
            print(f"Expected: 0x{expected:08X}")
            print(f"Actual:   0x{actual:08X}")
            break

PASS: Program write/readback verification succeeded.


In [22]:
import time

# Memory offsets
DATA_ARRAY_OFFSET = 0x0000
DONE_OFFSET = 0x1000

# CPU control values
CPU_STOP = 0x00000000
CPU_START = 0x00000001

# Verification settings
ARRAY_LENGTH = 32
DONE_MAGIC = 0xCAFEBABE

TIMEOUT_SECONDS = 10.0
POLL_INTERVAL_SECONDS = 0.01


def to_uint32(value):
    """
    Convert a Python signed integer to a 32-bit memory word.
    将 Python 有符号整数转换为 32-bit 内存格式。
    """
    return int(value) & 0xFFFFFFFF


def to_int32(value):
    """
    Interpret a 32-bit memory word as a signed integer.
    将读取的 32-bit 数值转换回有符号整数。
    """
    value = int(value) & 0xFFFFFFFF

    if value >= 0x80000000:
        return value - 0x100000000

    return value


def stop_cpu():
    reset_mmio.write(0x00, CPU_STOP)
    time.sleep(0.01)


def start_cpu():
    reset_mmio.write(0x00, CPU_START)

In [23]:
input_data = list(range(31, -1, -1))
expected_result = sorted(input_data)

print("Input:   ", input_data)
print("Expected:", expected_result)

assert len(input_data) == ARRAY_LENGTH

Input:    [31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
Expected: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


In [24]:
# Keep the CPU stopped while modifying memory
stop_cpu()

# Clear stale DONE value from previous executions
data_mmio.write(DONE_OFFSET, 0x00000000)

# Convert values to 32-bit words
encoded_input = [
    to_uint32(value)
    for value in input_data
]

# Write 32 integers to Data BRAM
write_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    encoded_input
)

print("CPU is held in reset.")
print("DONE flag cleared.")
print(f"{len(encoded_input)} integers written to Data BRAM.")

CPU is held in reset.
DONE flag cleared.
32 integers written to Data BRAM.


In [25]:
raw_input_readback = read_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    ARRAY_LENGTH
)

input_readback = [
    to_int32(word)
    for word in raw_input_readback
]

done_before_start = int(
    data_mmio.read(DONE_OFFSET)
) & 0xFFFFFFFF

print("Written input: ", input_data)
print("BRAM readback:", input_readback)
print(f"DONE before start: 0x{done_before_start:08X}")

if input_readback != input_data:
    for index, (expected, actual) in enumerate(
        zip(input_data, input_readback)
    ):
        if expected != actual:
            raise RuntimeError(
                f"Data mismatch at index {index}: "
                f"expected {expected}, got {actual}"
            )

if done_before_start != 0:
    raise RuntimeError(
        f"DONE was not cleared: 0x{done_before_start:08X}"
    )

print("PASS: Input data and DONE flag verified.")

Written input:  [31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
BRAM readback: [31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
DONE before start: 0x00000000
PASS: Input data and DONE flag verified.


In [26]:
def wait_for_done(
    timeout_seconds=TIMEOUT_SECONDS,
    poll_interval_seconds=POLL_INTERVAL_SECONDS
):
    """
    Wait until the RISC-V program writes CAFEBABE to DONE.
    等待 RISC-V 程序将 CAFEBABE 写入 DONE 地址。
    """
    start_time = time.monotonic()
    poll_count = 0
    last_done_value = 0

    while True:
        last_done_value = int(
            data_mmio.read(DONE_OFFSET)
        ) & 0xFFFFFFFF

        poll_count += 1

        if last_done_value == DONE_MAGIC:
            elapsed_seconds = time.monotonic() - start_time

            return {
                "elapsed_seconds": elapsed_seconds,
                "poll_count": poll_count,
                "done_value": last_done_value,
            }

        elapsed_seconds = time.monotonic() - start_time

        if elapsed_seconds >= timeout_seconds:
            raise TimeoutError(
                f"CPU did not finish within "
                f"{timeout_seconds:.2f} seconds. "
                f"Last DONE value: 0x{last_done_value:08X}"
            )

        time.sleep(poll_interval_seconds)

In [27]:
print("Starting RISC-V CPU...")

start_cpu()

try:
    completion = wait_for_done()

    print(
        f"DONE detected: "
        f"0x{completion['done_value']:08X}"
    )

    print(
        f"Elapsed time: "
        f"{completion['elapsed_seconds']:.6f} seconds"
    )

    print(
        f"Polling count: "
        f"{completion['poll_count']}"
    )

finally:
    # Always stop the CPU, even if timeout or exception occurs
    stop_cpu()
    print("CPU stopped.")

Starting RISC-V CPU...
DONE detected: 0xCAFEBABE
Elapsed time: 0.000058 seconds
Polling count: 1
CPU stopped.


In [28]:
final_done_value = int(
    data_mmio.read(DONE_OFFSET)
) & 0xFFFFFFFF

print(f"Final DONE value: 0x{final_done_value:08X}")

if final_done_value != DONE_MAGIC:
    raise RuntimeError(
        f"Incorrect DONE value: 0x{final_done_value:08X}"
    )

print("PASS: RISC-V program reported completion.")

Final DONE value: 0xCAFEBABE
PASS: RISC-V program reported completion.


In [29]:
raw_hardware_result = read_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    ARRAY_LENGTH
)

hardware_result = [
    to_int32(word)
    for word in raw_hardware_result
]

print("Hardware result:")
print(hardware_result)

Hardware result:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


In [30]:
print("Input:   ", input_data)
print("Expected:", expected_result)
print("Hardware:", hardware_result)

if hardware_result == expected_result:
    print("\nSUCCESS: Hardware sorting passed")

else:
    print("\nFAILURE: Hardware sorting mismatch")

    for index, (expected, actual) in enumerate(
        zip(expected_result, hardware_result)
    ):
        if expected != actual:
            print(f"First mismatch at index {index}")
            print(f"Expected: {expected}")
            print(f"Actual:   {actual}")
            print(
                f"Raw hardware word: "
                f"0x{raw_hardware_result[index]:08X}"
            )
            break

Input:    [31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
Expected: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Hardware: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]

SUCCESS: Hardware sorting passed


In [32]:
signed_input_data = [
    12, -4, 7, -20,
    7, 0, 31, -1,
    100, -100, 5, 5,
    -8, 42, 3, -50,
    20, -2, 18, -18,
    9, 9, -9, 1,
    200, -200, 15, -15,
    6, -6, 2, -3,
]

assert len(signed_input_data) == 32

signed_expected_result = sorted(signed_input_data)

print("Input:   ", signed_input_data)
print("Expected:", signed_expected_result)

Input:    [12, -4, 7, -20, 7, 0, 31, -1, 100, -100, 5, 5, -8, 42, 3, -50, 20, -2, 18, -18, 9, 9, -9, 1, 200, -200, 15, -15, 6, -6, 2, -3]
Expected: [-200, -100, -50, -20, -18, -15, -9, -8, -6, -4, -3, -2, -1, 0, 1, 2, 3, 5, 5, 6, 7, 7, 9, 9, 12, 15, 18, 20, 31, 42, 100, 200]


In [33]:
# Stop CPU before changing memory
stop_cpu()

# Clear previous completion flag
data_mmio.write(DONE_OFFSET, 0x00000000)

# Convert signed Python integers into 32-bit memory words
encoded_signed_input = [
    to_uint32(value)
    for value in signed_input_data
]

write_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    encoded_signed_input
)

print("Signed input written to Data BRAM.")
print("DONE flag cleared.")

Signed input written to Data BRAM.
DONE flag cleared.


In [35]:
raw_signed_readback = read_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    ARRAY_LENGTH
)

signed_readback = [
    to_int32(word)
    for word in raw_signed_readback
]

done_before_start = (
    int(data_mmio.read(DONE_OFFSET))
    & 0xFFFFFFFF
)

print("Original input: ", signed_input_data)
print("BRAM readback: ", signed_readback)
print(f"DONE before start: 0x{done_before_start:08X}")

if signed_readback != signed_input_data:
    for index, (expected, actual) in enumerate(
        zip(signed_input_data, signed_readback)
    ):
        if expected != actual:
            raise RuntimeError(
                f"Input mismatch at index {index}: "
                f"expected {expected}, got {actual}"
            )

if done_before_start != 0:
    raise RuntimeError(
        f"DONE was not cleared: 0x{done_before_start:08X}"
    )

print("PASS: Signed input readback succeeded.")

Original input:  [12, -4, 7, -20, 7, 0, 31, -1, 100, -100, 5, 5, -8, 42, 3, -50, 20, -2, 18, -18, 9, 9, -9, 1, 200, -200, 15, -15, 6, -6, 2, -3]
BRAM readback:  [12, -4, 7, -20, 7, 0, 31, -1, 100, -100, 5, 5, -8, 42, 3, -50, 20, -2, 18, -18, 9, 9, -9, 1, 200, -200, 15, -15, 6, -6, 2, -3]
DONE before start: 0x00000000
PASS: Signed input readback succeeded.


In [36]:
print("Starting signed-integer sorting test...")

start_cpu()

try:
    completion = wait_for_done(
        timeout_seconds=TIMEOUT_SECONDS
    )

    print(
        f"DONE detected: "
        f"0x{completion['done_value']:08X}"
    )

    print(
        f"Elapsed time: "
        f"{completion['elapsed_seconds']:.6f} seconds"
    )

finally:
    stop_cpu()
    print("CPU stopped.")

Starting signed-integer sorting test...
DONE detected: 0xCAFEBABE
Elapsed time: 0.000062 seconds
CPU stopped.


In [37]:
raw_signed_hardware_result = read_words(
    data_mmio,
    DATA_ARRAY_OFFSET,
    ARRAY_LENGTH
)

signed_hardware_result = [
    to_int32(word)
    for word in raw_signed_hardware_result
]

print("Input:   ", signed_input_data)
print("Expected:", signed_expected_result)
print("Hardware:", signed_hardware_result)

if signed_hardware_result == signed_expected_result:
    print("\nSUCCESS: Signed hardware sorting passed")

else:
    print("\nFAILURE: Signed hardware sorting mismatch")

    for index, (expected, actual) in enumerate(
        zip(
            signed_expected_result,
            signed_hardware_result
        )
    ):
        if expected != actual:
            print(f"First mismatch at index {index}")
            print(f"Expected: {expected}")
            print(f"Actual:   {actual}")
            print(
                "Raw hardware word: "
                f"0x{raw_signed_hardware_result[index]:08X}"
            )
            break

Input:    [12, -4, 7, -20, 7, 0, 31, -1, 100, -100, 5, 5, -8, 42, 3, -50, 20, -2, 18, -18, 9, 9, -9, 1, 200, -200, 15, -15, 6, -6, 2, -3]
Expected: [-200, -100, -50, -20, -18, -15, -9, -8, -6, -4, -3, -2, -1, 0, 1, 2, 3, 5, 5, 6, 7, 7, 9, 9, 12, 15, 18, 20, 31, 42, 100, 200]
Hardware: [-200, -100, -50, -20, -18, -15, -9, -8, -6, -4, -3, -2, -1, 0, 1, 2, 3, 5, 5, 6, 7, 7, 9, 9, 12, 15, 18, 20, 31, 42, 100, 200]

SUCCESS: Signed hardware sorting passed
